In [1]:
import pandas as pd

# Load the core tables
races = pd.read_csv('../data/races.csv')
drivers = pd.read_csv('../data/drivers.csv')
constructors = pd.read_csv('../data/constructors.csv')
results = pd.read_csv('../data/results.csv')

# Quick look at each
for name, df in [('races', races), ('drivers', drivers), ('constructors', constructors), ('results', results)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print()

--- races ---
Shape: (1125, 18)
Columns: ['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time']

--- drivers ---
Shape: (861, 9)
Columns: ['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob', 'nationality', 'url']

--- constructors ---
Shape: (212, 5)
Columns: ['constructorId', 'constructorRef', 'name', 'nationality', 'url']

--- results ---
Shape: (26759, 18)
Columns: ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']



In [2]:
# See full results columns
print("Full results columns:", list(results.columns))
print()

# Load two more key tables
lap_times = pd.read_csv('../data/lap_times.csv')
qualifying = pd.read_csv('../data/qualifying.csv')

for name, df in [('lap_times', lap_times), ('qualifying', qualifying)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print()

# Peek at actual rows to understand the data, not just column names
print(results.head(3))

Full results columns: ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']

--- lap_times ---
Shape: (589081, 6)
Columns: ['raceId', 'driverId', 'lap', 'position', 'time', 'milliseconds']

--- qualifying ---
Shape: (10494, 9)
Columns: ['qualifyId', 'raceId', 'driverId', 'constructorId', 'number', 'position', 'q1', 'q2', 'q3']

   resultId  raceId  driverId  constructorId number  grid position  \
0         1      18         1              1     22     1        1   
1         2      18         2              2      3     5        2   
2         3      18         3              3      7     7        3   

  positionText  positionOrder  points  laps         time milliseconds  \
0            1              1    10.0    58  1:34:50.616      5690616   
1            2              2     8.0    58       +5.478      5696094

In [3]:
# Check for the F1-famous "\N" missing value markers
print("Unique position values (sample):", results['position'].unique()[:20])
print()
print("Any '\\\\N' in position?", (results['position'] == r'\N').sum())
print()

# Check status.csv - this explains WHY a driver has that result
status = pd.read_csv('../data/status.csv')
print("--- status ---")
print(status.head(15))
print()

# Check data types pandas actually inferred
print(results.dtypes)

Unique position values (sample): <ArrowStringArray>
[ '1',  '2',  '3',  '4',  '5',  '6',  '7',  '8', '\N',  '9', '10', '11', '12',
 '13', '14', '15', '16', '17', '18', '19']
Length: 20, dtype: str

Any '\\N' in position? 10953

--- status ---
    statusId        status
0          1      Finished
1          2  Disqualified
2          3      Accident
3          4     Collision
4          5        Engine
5          6       Gearbox
6          7  Transmission
7          8        Clutch
8          9    Hydraulics
9         10    Electrical
10        11        +1 Lap
11        12       +2 Laps
12        13       +3 Laps
13        14       +4 Laps
14        15       +5 Laps

resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number                 str
grid                 int64
position               str
positionText           str
positionOrder        int64
points             float64
laps                 int64
time                   str


In [4]:
# Confirm positionOrder is always clean/numeric
print("positionOrder dtype:", results['positionOrder'].dtype)
print("Any \\N in positionOrder?", (results['positionOrder'].astype(str) == r'\N').sum())
print()

# Quick check on remaining tables we haven't peeked at yet
driver_standings = pd.read_csv('../data/driver_standings.csv')
constructor_standings = pd.read_csv('../data/constructor_standings.csv')
circuits = pd.read_csv('../data/circuits.csv')

for name, df in [('driver_standings', driver_standings), ('constructor_standings', constructor_standings), ('circuits', circuits)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print()

positionOrder dtype: int64
Any \N in positionOrder? 0

--- driver_standings ---
Shape: (34863, 7)
Columns: ['driverStandingsId', 'raceId', 'driverId', 'points', 'position', 'positionText', 'wins']

--- constructor_standings ---
Shape: (13391, 7)
Columns: ['constructorStandingsId', 'raceId', 'constructorId', 'points', 'position', 'positionText', 'wins']

--- circuits ---
Shape: (77, 9)
Columns: ['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt', 'url']



## Data Dictionary & Known Quirks — F1 Dataset

### Core tables and how they link
- **races** (1,125 rows) — one row per Grand Prix. Key: `raceId`
- **drivers** (861 rows) — one row per driver. Key: `driverId`
- **constructors** (212 rows) — one row per team. Key: `constructorId`
- **results** (26,759 rows) — one row per driver per race. **Central table** —
  links to races via `raceId`, drivers via `driverId`, constructors via `constructorId`
- **lap_times** (589,081 rows) — one row per driver per lap per race. Links via `raceId` + `driverId`
- **qualifying** (10,494 rows) — qualifying session results. Links via `raceId` + `driverId`
- **driver_standings** (34,863 rows) — championship standings snapshot after each race
- **constructor_standings** (13,391 rows) — team championship standings snapshot after each race
- **circuits** (77 rows) — track metadata incl. lat/lng. Key: `circuitId`
- **status** — lookup table explaining `statusId` (1 = Finished, 2-10ish = various DNF reasons, 11+ = finished but lapped e.g. "+1 Lap")

### Known data quirks (important for query logic)
1. **`results.position` is stored as TEXT, not numeric** — contains `'\N'` for
   non-finishers (~41% of all rows). **Do not** sort/aggregate on this directly.
   → Use **`positionOrder`** instead (confirmed clean `int64`, zero missing values) for anything race-finish-order related.
2. **`results.time`, `fastestLapTime`, `fastestLapSpeed` are stored as TEXT**
   (e.g. `"1:34:50.616"` or `"+5.478"`) — not directly usable in math.
   → Use **`milliseconds`** (numeric) for time-based calculations instead.
3. **`statusId` explains the *why*** behind a result — join to `status.csv` to
   distinguish "Finished," "Accident," "Engine failure," vs. "+1 Lap" (finished, just not on lead lap).
4. Missing-value marker across this dataset is the **literal string `'\N'`**,
   not pandas' default `NaN` — needs explicit handling when the schema inspector profiles a column.

In [6]:
import duckdb
import os

# Connect to an in-memory DuckDB instance for now (exploration only)
con = duckdb.connect(database=':memory:')

data_dir = '../data'
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files: {csv_files}\n")

# Register each CSV as a DuckDB table, named after the file
for csv_file in csv_files:
    table_name = csv_file.replace('.csv', '')
    path = os.path.join(data_dir, csv_file)
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{path}')
    """)
    print(f"Loaded table: {table_name}")

print("\n--- All tables in DuckDB ---")
print(con.execute("SHOW TABLES").fetchdf())

# Test query using a real 3-table join
test_query = """
    SELECT c.name AS constructor, COUNT(*) AS wins
    FROM results r
    JOIN races ra ON r.raceId = ra.raceId
    JOIN constructors c ON r.constructorId = c.constructorId
    WHERE ra.year = 2015 AND r.positionOrder = 1
    GROUP BY c.name
    ORDER BY wins DESC
"""
print("\n--- Test query: 2015 wins by constructor ---")
print(con.execute(test_query).fetchdf())

Found 14 CSV files: ['circuits.csv', 'constructors.csv', 'constructor_results.csv', 'constructor_standings.csv', 'drivers.csv', 'driver_standings.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv', 'races.csv', 'results.csv', 'seasons.csv', 'sprint_results.csv', 'status.csv']

Loaded table: circuits
Loaded table: constructors
Loaded table: constructor_results
Loaded table: constructor_standings
Loaded table: drivers
Loaded table: driver_standings
Loaded table: lap_times
Loaded table: pit_stops
Loaded table: qualifying
Loaded table: races
Loaded table: results
Loaded table: seasons
Loaded table: sprint_results
Loaded table: status

--- All tables in DuckDB ---
                     name
0                circuits
1     constructor_results
2   constructor_standings
3            constructors
4        driver_standings
5                 drivers
6               lap_times
7               pit_stops
8              qualifying
9                   races
10                results
11            